# 📐 Chapter 9: Orthogonal Matrices and QR Decomposition

---

## 9.1 Introduction: The Need for Matrix Decompositions

In the previous chapter, we learned that computing the inverse of a matrix directly (using $A^{-1}$) can be computationally expensive and dangerously prone to numerical instability (floating-point rounding errors).

In applied linear algebra and data science, we almost never compute the inverse directly. Instead, we use **Matrix Decompositions**. A matrix decomposition breaks a complex matrix down into simpler, foundational matrices—much like factoring the number 12 into $2 \times 2 \times 3$. 

In this chapter, we explore one of the most important computational decompositions: the **QR Decomposition**. Before understanding QR, we must first understand the elegant mathematical "superpowers" of **Orthogonal Matrices**, which form the 'Q' in the decomposition.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(42)
print("Scientific libraries successfully imported for Chapter 9.")

## 9.2 Orthogonal Matrices (The 'Q' Matrix)

An **Orthogonal Matrix** (usually denoted as $\mathbf{Q}$) is a square matrix whose columns and rows are mutually orthogonal unit vectors (also known as an *orthonormal* set of vectors). 

Orthogonal matrices possess a mathematical superpower that makes them the holy grail of linear algebra algorithms:
**Their transpose is exactly equal to their inverse.**

$$ \mathbf{Q}^T \mathbf{Q} = \mathbf{I} $$
$$ \mathbf{Q}^{-1} = \mathbf{Q}^T $$

Because computing a transpose takes practically zero computational effort (it just re-indexes memory) compared to computing an inverse (which requires complex elimination steps), orthogonal matrices allow us to "invert" systems instantly and safely. Furthermore, multiplying a vector by an orthogonal matrix strictly rotates or reflects it—it never stretches or shrinks it (it preserves the L2 norm/magnitude).

In [ ]:
# Let's create an Orthogonal Matrix. 
# A standard 2D rotation matrix is inherently orthogonal.
theta = np.pi / 4  # 45 degrees
Q = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

print("Orthogonal Matrix Q (Rotation by 45 deg):\n", np.round(Q, 3))

# 1. Verify that Q^T @ Q equals the Identity Matrix (I)
identity_check = Q.T @ Q
print("\nVerification of Q^T @ Q (Should be Identity Matrix):\n", np.round(identity_check, 3))

# 2. Verify Norm Preservation
v = np.array([3, 4]) # Magnitude is 5
v_transformed = Q @ v

print(f"\nOriginal vector length: {np.linalg.norm(v)}")
print(f"Length after Q transformation: {np.linalg.norm(v_transformed)}")
print("Notice the length is perfectly preserved! Q only rotated the vector.")

## 9.3 The Gram-Schmidt Process

If orthogonal matrices are so amazing, how do we turn an ordinary, messy matrix $\mathbf{A}$ into a beautiful orthogonal matrix $\mathbf{Q}$? We use the **Gram-Schmidt Process**.

The algorithm works step-by-step across the columns of a matrix:
1. **Start:** Take the first column vector. Normalize it (divide by its length so its magnitude becomes 1). This is your first orthogonal vector, $q_1$.
2. **Project and Subtract:** Take the second column vector. Find how much of it points in the same direction as $q_1$ (using the dot product projection). Subtract that overlapping portion. What remains is perfectly orthogonal to $q_1$. Normalize it to get $q_2$.
3. **Repeat:** For the third vector, subtract its projections onto both $q_1$ and $q_2$. Normalize to get $q_3$.
4. Continue until all columns are strictly orthogonal and have a length of 1.

In [ ]:
# A manual, step-by-step implementation of Gram-Schmidt for 2 vectors
v1 = np.array([3, 1], dtype=float)
v2 = np.array([2, 2], dtype=float)

# Step 1: Normalize v1 to get q1
q1 = v1 / np.linalg.norm(v1)

# Step 2: Subtract the projection of v2 onto q1
projection_v2_q1 = np.dot(v2, q1) * q1
v2_orthogonal = v2 - projection_v2_q1

# Step 3: Normalize to get q2
q2 = v2_orthogonal / np.linalg.norm(v2_orthogonal)

# Construct the Orthogonal Matrix Q
Q_manual = np.column_stack((q1, q2))
print("Manually computed Q via Gram-Schmidt:\n", np.round(Q_manual, 3))

# Verify orthogonality
print("\nCheck Q^T @ Q:\n", np.round(Q_manual.T @ Q_manual, 3))

## 9.4 QR Decomposition

The Gram-Schmidt process is the mathematical engine behind **QR Decomposition**. Any matrix $\mathbf{A}$ (even non-square, rectangular matrices) can be factored into two matrices:
$$ \mathbf{A} = \mathbf{Q} \mathbf{R} $$

- $\mathbf{Q}$: An orthogonal matrix (containing orthonormal columns) holding the pure directional spaces.
- $\mathbf{R}$: An **Upper Triangular matrix** (all zeros below the main diagonal). This matrix holds the "leftover" magnitudes and non-orthogonal components that were stripped away during the Gram-Schmidt process.

**Why is QR Decomposition so crucial in Machine Learning?**
When solving $\mathbf{A}\mathbf{x} = \mathbf{b}$ (like finding the weights in Linear Regression), we can substitute $\mathbf{A}$ with $\mathbf{Q}\mathbf{R}$:
$$ \mathbf{Q}\mathbf{R}\mathbf{x} = \mathbf{b} $$

Multiply both sides by $\mathbf{Q}^T$ (which is $\mathbf{Q}^{-1}$):
$$ \mathbf{R}\mathbf{x} = \mathbf{Q}^T \mathbf{b} $$

Because $\mathbf{R}$ is upper triangular, this new system can be solved instantly from bottom-to-top using simple back-substitution, completely avoiding the dangerous standard matrix inverse!

In [ ]:
# Create a random 3x3 matrix (Messy, non-orthogonal)
A = np.random.randint(1, 10, size=(3, 3))

# Perform QR Decomposition using NumPy
Q, R = np.linalg.qr(A)

print("Original Matrix A:")
print(A)

print("\nOrthogonal Matrix Q (Notice it has decimals and negative signs):")
print(np.round(Q, 3))

print("\nUpper Triangular Matrix R (Notice the zeros below the diagonal):")
print(np.round(R, 3))

# 1. Verify A = QR
A_reconstructed = Q @ R
print("\nVerify Q @ R recovers original A:")
print(np.round(A_reconstructed))

# 2. Verify Q is indeed orthogonal
print("\nVerify Q is orthogonal (Q^T @ Q = I):")
print(np.round(Q.T @ Q, 3))

## 9.5 The Geometrics of QR Decomposition

To build geometric intuition:
- Matrix $\mathbf{A}$ rotates, scales, and shears space all at once.
- Matrix $\mathbf{Q}$ handles the **pure rotation/reflection** part of the transformation.
- Matrix $\mathbf{R}$ handles the **scaling and shearing** part of the transformation.

QR Decomposition gracefully separates the orientation of the data from the magnitude/distortion of the data.

In [ ]:
# Let's visualize how the R matrix is upper triangular in larger matrices
A_large = np.random.randn(10, 10)
Q_large, R_large = np.linalg.qr(A_large)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(Q_large, cmap='gray')
plt.title("Q Matrix (Dense Orthogonal)")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(R_large, cmap='gray')
plt.title("R Matrix (Upper Triangular)")
plt.colorbar()

plt.tight_layout()
plt.show()
print("The black triangle in the R matrix confirms that all values below the diagonal are exactly zero.")

## 9.6 Chapter Summary

In Chapter 9, we explored one of the most practical toolkits in applied linear algebra:
- **Orthogonal Matrices (Q):** Their transpose is exactly their inverse. They act purely as rigid rotations, preserving vector distances and angles.
- **Gram-Schmidt Process:** The algorithmic engine that purifies a set of vectors into an orthogonal, normalized set.
- **QR Decomposition:** Factoring a messy matrix into $\mathbf{Q}$ (orthogonal) and $\mathbf{R}$ (upper triangular). 
- **Why it matters:** It is the gold standard for solving systems of linear equations ($Ax = b$) and Ordinary Least Squares safely inside computers, completely avoiding the numeric disaster of directly calculating $A^{-1}$.